In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

!pip install category_encoders
!pip install optuna
from category_encoders import TargetEncoder
import optuna
import lightgbm as lgb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 11.1 MB/s eta 0:00:00


In [ ]:
train = pd.read_csv('train.csv')

In [ ]:
train

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439135,439135,D755,MEDIUM,Miami Grand Prix,2023,0,49,2,8.0,17,92.638,-0.076,-15.859,0.859649,0.0,0.0
439136,439136,D731,MEDIUM,Miami Grand Prix,2023,0,49,2,5.0,1,85.890,-0.083,-4.907,0.859649,0.0,0.0
439137,439137,D716,MEDIUM,Miami Grand Prix,2023,0,49,2,18.0,1,91.644,-0.182,-56.371,0.942308,0.0,0.0
439138,439138,D665,HARD,Abu Dhabi Grand Prix,2023,0,48,3,10.0,1,89.947,-0.001,-20.721,0.827586,1.0,0.0


In [ ]:
print('Compound ::',train['Compound'].unique())
print('Race ::' , train['Race'].unique())

Compound :: ['HARD' 'MEDIUM' 'INTERMEDIATE' 'SOFT' 'WET']
Race :: ['Canadian Grand Prix' 'Dutch Grand Prix' 'Austrian Grand Prix'
 'Pre-Season Testing' 'Azerbaijan Grand Prix' 'Saudi Arabian Grand Prix'
 'Belgian Grand Prix' 'United States Grand Prix' 'Italian Grand Prix'
 'Hungarian Grand Prix' 'Japanese Grand Prix' 'São Paulo Grand Prix'
 'Bahrain Grand Prix' 'Las Vegas Grand Prix' 'Monaco Grand Prix'
 'British Grand Prix' 'Australian Grand Prix' 'Spanish Grand Prix'
 'Miami Grand Prix' 'French Grand Prix' 'Abu Dhabi Grand Prix'
 'Chinese Grand Prix' 'Mexico City Grand Prix' 'Emilia Romagna Grand Prix'
 'Singapore Grand Prix' 'Qatar Grand Prix']


In [ ]:
numaric_col = train.select_dtypes(exclude = ['object']).columns
object_col = train.select_dtypes(include = ['object']).columns
print("Numaric Col :",numaric_col)
print("Object Col :" , object_col)

Numaric Col : Index(['id', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
       'RaceProgress', 'Position_Change', 'PitNextLap'],
      dtype='object')
Object Col : Index(['Driver', 'Compound', 'Race'], dtype='object')


In [ ]:
for i,col in enumerate(numaric_col):
    print(col , ':' ,train[col].skew())

id : 1.0438934133511283e-15
Year : -0.0006106196835017434
PitStop : 2.1222974022518595
LapNumber : 0.6459650188747418
Stint : 1.1854786226203042
TyreLife : 1.0324483774971478
Position : 0.0700976445736444
LapTime (s) : 80.33432778187277
LapTime_Delta : -40.855060765987595
Cumulative_Degradation : 3.191500679087761
RaceProgress : 0.6999501428857346
Position_Change : -0.09271426725985092
PitNextLap : 1.5079803487341805


In [ ]:
train.drop(['id'], axis=1, inplace=True)

In [ ]:
train

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439135,D755,MEDIUM,Miami Grand Prix,2023,0,49,2,8.0,17,92.638,-0.076,-15.859,0.859649,0.0,0.0
439136,D731,MEDIUM,Miami Grand Prix,2023,0,49,2,5.0,1,85.890,-0.083,-4.907,0.859649,0.0,0.0
439137,D716,MEDIUM,Miami Grand Prix,2023,0,49,2,18.0,1,91.644,-0.182,-56.371,0.942308,0.0,0.0
439138,D665,HARD,Abu Dhabi Grand Prix,2023,0,48,3,10.0,1,89.947,-0.001,-20.721,0.827586,1.0,0.0


In [ ]:
def data_encode(df):
    df = df.copy()

    df = pd.get_dummies(df,columns=['Compound'],drop_first=False , dtype=int)

    return df

groups = train['Driver'].astype(str) + "_" + train['Race'].astype(str)

train = data_encode(train)

In [ ]:
train

,Driver,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET
0,D109,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0,1,0,0,0,0
1,D086,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0,1,0,0,0,0
2,ZON,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0,1,0,0,0,0
3,SPE,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0,0,0,1,0,0
4,D019,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439135,D755,Miami Grand Prix,2023,0,49,2,8.0,17,92.638,-0.076,-15.859,0.859649,0.0,0.0,0,0,1,0,0
439136,D731,Miami Grand Prix,2023,0,49,2,5.0,1,85.890,-0.083,-4.907,0.859649,0.0,0.0,0,0,1,0,0
439137,D716,Miami Grand Prix,2023,0,49,2,18.0,1,91.644,-0.182,-56.371,0.942308,0.0,0.0,0,0,1,0,0
439138,D665,Abu Dhabi Grand Prix,2023,0,48,3,10.0,1,89.947,-0.001,-20.721,0.827586,1.0,0.0,1,0,0,0,0


In [ ]:
driver_freq = train["Driver"].value_counts(normalize=True)
train["Driver_FE"] = train["Driver"].map(driver_freq)

race_freq = train["Race"].value_counts(normalize=True)
train["Race_FE"] = train["Race"].map(race_freq)

train.drop(columns=["Driver", "Race"], inplace=True)

In [ ]:
train.drop(columns=["Year"], inplace=True)

In [ ]:
train

,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET,Driver_FE,Race_FE
0,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0,1,0,0,0,0,0.002345,0.048768
1,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0,1,0,0,0,0,0.002671,0.055704
2,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0,1,0,0,0,0,0.003336,0.048329
3,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0,0,0,1,0,0,0.003261,0.051218
4,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0,1,0,0,0,0,0.003316,0.027613
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439135,0,49,2,8.0,17,92.638,-0.076,-15.859,0.859649,0.0,0.0,0,0,1,0,0,0.000002,0.042948
439136,0,49,2,5.0,1,85.890,-0.083,-4.907,0.859649,0.0,0.0,0,0,1,0,0,0.000002,0.042948
439137,0,49,2,18.0,1,91.644,-0.182,-56.371,0.942308,0.0,0.0,0,0,1,0,0,0.000002,0.042948
439138,0,48,3,10.0,1,89.947,-0.001,-20.721,0.827586,1.0,0.0,1,0,0,0,0,0.000005,0.037407


In [ ]:
train.to_csv('processed_train_data.csv', index=False)

from google.colab import files
files.download('processed_train_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>